# Model Training and Evaluation - Credit Card Fraud Detection

This notebook implements the model training pipeline, compares different classification algorithms, handles dataset imbalances using resampling techniques, tunes hyperparameters, and evaluates performance on a holdout test set using metrics suitable for imbalanced datasets.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Set style
sns.set_theme(style="darkgrid")

## 1. Data Preparation and Preprocessing

In [ ]:
data_path = os.path.join("..", "data", "creditcard.csv")
if not os.path.exists(data_path):
    raise FileNotFoundError(f"Dataset not found. Please run the training script src/train.py first.")

df = pd.read_csv(data_path)

# Deduplicate
df = df.drop_duplicates().reset_index(drop=True)

X = df.drop(columns=['Class'])
y = df['Class']

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature scaling for 'Time' and 'Amount'
scaler = StandardScaler()
cols_to_scale = ['Time', 'Amount']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

## 2. Handling Imbalance (Resampling)

In [ ]:
# Apply SMOTE (Over-sampling)
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled.values, y_train.values)

# Apply Random Under-sampling
rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train_scaled.values, y_train.values)

print(f"Original Train Set Class Distribution: Genuine={sum(y_train==0)}, Fraud={sum(y_train==1)}")
print(f"SMOTE Train Set Class Distribution: Genuine={sum(y_train_smote==0)}, Fraud={sum(y_train_smote==1)}")
print(f"RUS Train Set Class Distribution: Genuine={sum(y_train_rus==0)}, Fraud={sum(y_train_rus==1)}")

## 3. Train and Compare Models

In [ ]:
# Let's load the model comparisons generated from src/train.py
results_path = os.path.join("..", "models", "comparison_results.csv")
if os.path.exists(results_path):
    comp_results = pd.read_csv(results_path)
    print("Loaded Model Comparison Results:")
    display(comp_results.sort_values(by='F1-Score', ascending=False))
else:
    print("Comparison results not found. Please run 'python src/train.py' in the command line to train all models and produce comparisons.")

In [ ]:
# Visual comparison of models F1-Scores across different sampling strategies
if os.path.exists(results_path):
    plt.figure(figsize=(12, 6))
    sns.barplot(x='Model', y='F1-Score', hue='Sampling', data=comp_results)
    plt.title('Model F1-Score Comparison by Resampling Strategy')
    plt.xticks(rotation=15)
    plt.ylabel('F1-Score (Holdout Test Set)')
    plt.legend(title='Sampling Strategy')
    plt.tight_layout()
    plt.show()

### Insight: Resampling Strategy Performance
- **Original (Imbalanced Data)**: Models trained on original data show high Precision but often suffer on Recall, as they prioritize predicting the majority class (Genuine).
- **SMOTE (Synthetic Over-sampling)**: SMOTE increases the training size dramatically. Models trained on SMOTE show significantly higher Recall but can experience a drop in Precision (more false positives) because synthetic fraud transactions cover regions that slightly overlap genuine data.
- **Random Under-Sampling (RUS)**: RUS balances the classes by throwing away the majority of genuine data. While training is extremely fast, models often suffer from a severe drop in Precision because they lose valuable context about the genuine class distributions, resulting in excessive false alarms.

## 4. Hyperparameter Tuning & Model Serialization

In [ ]:
# Load final model metadata
import joblib
meta_path = os.path.join("..", "models", "best_model_meta.pkl")
if os.path.exists(meta_path):
    best_meta = joblib.load(meta_path)
    print("Best Performing Model Metadata:")
    print(f"  Model Name       : {best_meta['model_name']}")
    print(f"  Sampling Strategy: {best_meta['sampling_strategy']}")
    print(f"  Best Hyperparams : {best_meta['best_params']}")
    print(f"  Final Metrics    :")
    for k, v in best_meta['metrics'].items():
        print(f"    - {k:10}: {v:.4f}")
else:
    print("Best model metadata not found. Please run training script first.")

## 5. Model Interpretability (SHAP & Feature Importance)

In [ ]:
# If SHAP is saved, display feature importance
shap_values_path = os.path.join("..", "models", "shap_values.pkl")
shap_sample_path = os.path.join("..", "models", "shap_sample.pkl")

if os.path.exists(shap_values_path) and os.path.exists(shap_sample_path):
    import shap
    shap_values = joblib.load(shap_values_path)
    shap_sample = joblib.load(shap_sample_path)
    
    print("Displaying SHAP Summary Plot...")
    plt.figure()
    shap.summary_plot(shap_values, shap_sample, show=False)
    plt.title('SHAP Values Summary Plot', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("SHAP values not pre-computed or SHAP is unavailable in the environment. Feature importance plots will be displayed in the Streamlit app instead.")